# Лабораторная работа:
# Medical Deep Research Agent

## Цель работы
Создать мультиагентную систему для глубоких медицинских исследований, которая:
1. Использует **MCP-сервера** для поиска научных статей (medRxiv, arXiv).
2. Анализирует найденные материалы.
3. Генерирует финальный отчет в формате **PDF**.

## Задание
- Настроить подключение к Yandex Cloud и OpenAI Agents SDK.
- Подключить MCP-сервера для источников данных.
- Реализовать инструмент для компиляции/генерации PDF.
- Спроектировать архитектуру агентов (Writer, Researcher, Analyst и др.).
- Запустить исследование на медицинскую тему.


In [ ]:
# Установка зависимостей
%pip install --quiet openai-agents mcp python-dotenv fpdf2 shwarsutils

## Часть 1: Инициализация и настройка
Используйте этот код для настройки соединения с Yandex Cloud.


In [ ]:
!curl -o .env {{url_of_dotenv_file}}

Используйте код ниже из ноутбука с примерами для настройки модели для работы с OpenAI Agents SDK.

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from agents import Agent, Runner, WebSearchTool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel
from openai import AsyncOpenAI

load_dotenv()

folder_id = os.environ.get("folder_id")
api_key = os.environ.get("api_key")

set_tracing_disabled(True)

yandex_client = AsyncOpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

model_uri = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

yandex_model = OpenAIResponsesModel(
    model=model_uri,
    openai_client=yandex_client
)

def printx(string):
    display(Markdown(string))

print(f"✅ Система готова к работе. Folder id: {folder_id[:8]}...")

✅ Система готова к работе. Folder id: b1gkp4q6...


## Часть 2: Мониторинг (Hooks)
Класс для отслеживания работы агентов в реальном времени. Взят из демонстрационного ноутбука и уже готов к использованию.


In [2]:
from agents import RunHooks, RunContextWrapper, Tool

class VerboseHooks(RunHooks):
    """
    Hooks для вывода действий агента в реальном времени.
    Показывает старт/завершение агентов, вызовы инструментов и handoffs.
    """
    async def on_agent_start(self, context: RunContextWrapper, agent):
        print(f"🤖 Агент запущен: {agent.name}")
    
    async def on_agent_end(self, context: RunContextWrapper, agent, output):
        output_preview = str(output)[:100] + "..." if len(str(output)) > 100 else str(output)
        print(f"✅ Агент завершён: {agent.name}")
        print(f"   └─ Результат: {output_preview}")
    
    async def on_tool_start(self, context: RunContextWrapper, agent, tool: Tool):
        # Локальные инструменты
        print(f"🔧 Вызов локального инструмента: {tool.name}")
    
    async def on_tool_end(self, context: RunContextWrapper, agent, tool: Tool, result):
        result_preview = str(result)[:80] + "..." if len(str(result)) > 80 else str(result)
        print(f"   └─ Результат: {result_preview}")

    async def on_handoff(self, context: RunContextWrapper, from_agent, to_agent):
        print(f"🔀 Handoff: {from_agent.name} → {to_agent.name}")
    
    async def on_llm_end(self, context: RunContextWrapper, agent, response):
        """Показывает вызовы hosted tools и MCP."""
        for item in response.output:
            item_type = getattr(item, 'type', None)
            if item_type == 'web_search_call':
                query = getattr(item, 'query', 'query hidden')
                print(f"🌐 Web Search: {query}")
            elif item_type == 'function_call':
                name = getattr(item, 'name', 'unknown')
                args = getattr(item, 'arguments', '')[:50]
                print(f"📡 MCP/Function call: {name}({args}...)")

verbose_hooks = VerboseHooks()
print("✅ Hooks созданы")

✅ Hooks созданы


## Часть 3: Подключение MCP-серверов

Для лабораторной работы мы будем использовать MCP-сервера, рассмотренные нами ранее и развёрнутые удалённо на виртуальной машине Yandex Cloud, либо где-то в другом месте. 

Вам нужно получить URL адреса (SSE) для этих серверов у преподавателя или развернуть их самостоятельно.

Реализация MCP-серверов находится в директории [servers](../../5-mcp/servers/). Вы можете запустить отдельно [arXiv MCP Server](../../5-mcp/servers/arxiv_mcp.py) или [medrXiv MCP Server](../../5-mcp/servers/medrxiv_mcp.py), либо запустить [маршрутизатор MCP](../../5-mcp/servers/mcp_gateway.py):
```
python mcp_gateway.py --mode multi
```
Эта команда запустить все имеющиеся в текущей диретории MCP-сервера на последовательных портах, начиная с 8000.

Опишите конфигурацию MCP-серверов ниже:

In [ ]:
# Конфигурация инструментов MCP

from agents import HostedMCPTool

arxiv_tool_config = {
    "type": "mcp",
    "server_label": "arXiv-Research",
    "server_description": "Поиск научных статей по физике и CS на arXiv",
    "server_url": "http://cathy.ycloud.eazify.net:8000/sse",  # <-- Вставьте ваш URL
    "require_approval": "never"
}

arxiv_tool = HostedMCPTool(tool_config=arxiv_tool_config)

## Аналогичным образом опишите конфигурацию сервера для medrXiv

print("✅ Конфигурация MCP задана")

✅ Конфигурация MCP задана


Убедитесь, что MCP-сервер корректно вызывается:

In [1]:
# Вызовите MCP-сервера с помощью Responses API через client.responses.create

### Пример: Агент с доступом к arXiv

Создадим простого агента, чтобы проверить подключение.


In [2]:
# Создайте и запустите агента OpenAI Agents SDK с одним MCP инструментом


## Часть 4: Генерация отчета (PDF)

Агент должен уметь сохранять результаты исследования не просто в текст, а в красивый документ.
**Задача**: Реализуйте инструмент `generate_pdf_report`, который:
1. Принимает контент (возможно, в формате LaTeX или Markdown).
2. Генерирует PDF файл.
3. Возвращает путь к файлу.

### Решение

Мы используем библиотеку `fpdf2` — чистый Python, не требует установки LaTeX.
Агент будет передавать текст в формате **Markdown**, а функция распарсит его
и сгенерирует PDF с заголовками, списками и форматированием.

Для работы с кириллицей нам потребуется использовать шрифт:

In [4]:
from shwars.utils import download_file, unzip_file

download_file("https://font.download/dl/font/dejavu-sans.zip")
unzip_file("dejavu-sans.zip")

Опишем функцию генерации PDF-файла. Для удобства, мы предоставляем вам готовую реализацию этой функции:

In [11]:
from fpdf import FPDF
import re

def generate_pdf(content: str, filename: str = "report.pdf") -> str:
    pdf = FPDF()
    pdf.add_page()
    
    pdf.add_font("DejaVu", "", "DejaVuSans.ttf")
    pdf.add_font("DejaVu", "B", "DejaVuSans-Bold.ttf")
    pdf.set_font("DejaVu", size=11)

    for line in content.split("\n"):
        stripped = line.strip()
        if not stripped:
            pdf.ln(4)
            continue

        # Заголовки
        if stripped.startswith("### "):
            pdf.set_font("DejaVu", "B", 13)
            pdf.multi_cell(0, 7, stripped[4:])
            pdf.ln(2)
            pdf.set_font("DejaVu", "", 11)
        elif stripped.startswith("## "):
            pdf.set_font("DejaVu", "B", 15)
            pdf.multi_cell(0, 8, stripped[3:])
            pdf.ln(3)
            pdf.set_font("DejaVu", "", 11)
        elif stripped.startswith("# "):
            pdf.set_font("DejaVu", "B", 18)
            pdf.multi_cell(0, 10, stripped[2:])
            pdf.ln(4)
            pdf.set_font("DejaVu", "", 11)
        elif stripped.startswith("- ") or stripped.startswith("* "):
            # Списки
            text = stripped[2:]
            # Убираем Markdown-разметку для PDF
            text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
            pdf.cell(5)
            pdf.multi_cell(0, 6, f"• {text}")
            pdf.ln(1)
        elif stripped.startswith("---"):
            # Горизонтальная линия
            pdf.ln(3)
            y = pdf.get_y()
            pdf.line(10, y, 200, y)
            pdf.ln(3)
        else:
            # Обычный текст — убираем Markdown-разметку
            text = re.sub(r'\*\*(.*?)\*\*', r'\1', stripped)
            text = re.sub(r'\*(.*?)\*', r'\1', text)
            pdf.multi_cell(0, 6, text)
            pdf.ln(1)

    pdf.output(filename)

Убедимся, что она работает:

In [12]:
test_content = """# Тестовый отчет

## Введение
Это тестовый отчет для проверки генерации PDF.

### Ключевые находки
- **Первый пункт**: Тестирование работает корректно.
- **Второй пункт**: Кириллица поддерживается.

---

## Заключение
Инструмент генерации PDF готов к использованию.
"""

generate_pdf(test_content, "test_report.pdf")

Теперь опишем функцию генерации как инструмент для LLM:

In [3]:
# Оберните функцию `generate_pdf` в инструмент, чтобы агенты
# могли её вызывать. Функция будет сохранять PDF-отчёт на диск, и
# возвращать путь к файлу

## Часть 5: Инструмент заметок

По аналогии с демонстрационным ноутбуком, создадим инструмент для сохранения 
заметок в процессе исследования. Исследователь будет сохранять ключевые факты,
а писатель — использовать их для составления отчёта.

In [4]:
# Реализуйте инструмент для сохранения заметок
# Вы можете взять код из лекционного примера, или также использовать
# простой MCP-сервер дла заметок, или попробовать найти MCP-сервер для
# работы с записными книжками инструментов вроде Obsidian

## Часть 6: Архитектура агентов

Спроектируйте команду агентов. Рекомендуемые роли:
- **MedicalPlanner**: Планирует стратегию поиска.
- **LiteratureSearcher**: Использует MCP инструменты для поиска статей.
- **EvidenceAnalyst**: Анализирует найденные данные на предмет противоречий.
- **MedicalWriter**: Формирует структуру отчета и пишет LaTeX код.

Выберите паттерн координации:
- **Chain**: Последовательная передача.
- **Hub-and-Spoke**: Центральный координатор управляет процессом.
- **Network**: Свободное общение (сложнее в отладке).
- **Agent-as-Tool**: Центральный агнет-координатор использует под-агентов как инструменты, не передавая единый контекст беседы

In [5]:
# Опишите отдельных агентов и мультиагентную систему на их основе

## Часть 7: Запуск исследования
Запустите агента на сложной медицинской теме. Примеры:
- "Latest treatments for Alzheimer's disease in 2024-2025"
- "Comparing mRNA and protein-based vaccines efficacy data"


In [6]:
# Используйте реализацию с Hooks или с Streaming для контроля за тем,
# как проходит процесс взаимодействия между агентами

### Сохранённые заметки
Посмотрите, какие заметки были собраны в процессе исследования:

In [7]:
# Печать заметок
# Если вы использовали MCP-сервер, то придётся использовать LLM для доступа
# к нему.

---

## Заключение

В этой лабораторной работе вы построили **мультиагентную систему для медицинских исследований**. 

### Ключевые выводы

- **Декомпозиция задач** между агентами повышает качество каждого шага: планирование, поиск, анализ и написание выполняются специализированно.
- **MCP-протокол** обеспечивает стандартизированный доступ к внешним инструментам без необходимости писать кастомные интеграции.
- **Механизм заметок** позволяет агентам обмениваться информацией между шагами, формируя общую базу знаний.
- **Streaming Events** — удобная альтернатива Hooks для отслеживания прогресса в реальном времени.